In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

from fmri_fm_eval.datasets.registry import create_dataset

In [5]:
def get_dataset_info(dataset):
    splits = ["train", "validation", "test"]
    counts = [len(dataset[split]) for split in splits]
    total_count = sum(counts)
    split_counts = ":".join(f"{100 * count / total_count:.0f}" for count in counts)

    split_subs = {}
    for split, ds in dataset.items():
        subs = np.asarray(ds.dataset["sub"])[ds.indices]
        split_subs[split] = np.unique(subs)
    all_subs = np.unique(np.concatenate(list(split_subs.values())))

    sample_lengths = []
    sample_trs = []
    for split in splits:
        for sample in dataset[split]:
            bold = sample["bold"]
            tr = sample["tr"]
            sample_lengths.append(len(bold))
            sample_trs.append(tr)
    sample_length = np.mean(sample_lengths)
    sample_tr = np.mean(sample_trs)

    label_counts = {}
    for split in splits:
        for label, count in zip(dataset[split].labels, dataset[split].label_counts):
            label_counts[label] = label_counts.get(label, 0) + count
    n_classes = len(label_counts)
    majority_ratio = max(label_counts.values()) / sum(label_counts.values())

    record = {
        "subjects": len(all_subs),
        "samples": total_count,
        "sample length": f"{sample_length:.0f}",
        "sample tr": f"{sample_tr:.1f}s",
        "splits": split_counts,
        "num classes": n_classes,
        "majority ratio": f"{100 * majority_ratio:.0f}%",
        "labels": " ".join(f"{label}: {count}" for label, count in label_counts.items()),
    }
    return record

In [6]:
dataset = create_dataset("abide_age", "schaefer400")
print(dataset)
get_dataset_info(dataset)

{'train': HFDataset(
    dataset=Dataset({
    features: ['sub', 'site', 'dataset', 'path', 'n_frames', 'tr', 'bold', 'mean', 'std'],
    num_rows: 578
}),
    labels=['Q1' 'Q2' 'Q3'],
    counts=[198 198 182]
), 'validation': HFDataset(
    dataset=Dataset({
    features: ['sub', 'site', 'dataset', 'path', 'n_frames', 'tr', 'bold', 'mean', 'std'],
    num_rows: 124
}),
    labels=['Q1' 'Q2' 'Q3'],
    counts=[43 42 39]
), 'test': HFDataset(
    dataset=Dataset({
    features: ['sub', 'site', 'dataset', 'path', 'n_frames', 'tr', 'bold', 'mean', 'std'],
    num_rows: 124
}),
    labels=['Q1' 'Q2' 'Q3'],
    counts=[42 43 39]
)}


{'subjects': 826,
 'samples': 826,
 'sample length': '150',
 'sample tr': '2.0s',
 'splits': '70:15:15',
 'num classes': 3,
 'majority ratio': '34%',
 'labels': 'Q1: 283 Q2: 283 Q3: 260'}

In [7]:
DATASET_NAMES = {
    "abide_dx": "ABIDE Dx",
    "abide_age": "ABIDE Age",
    # "abide_sex": "ABIDE Sex",
    "adhd200_dx": "ADHD200 Dx",
    "adhd200_sex": "ADHD200 Sex",
    "adni_ad_vs_cn": "ADNI Dx",
    "adni_sex": "ADNI Sex",
    "ppmi_dx": "PPMI Dx",
    "ppmi_sex": "PPMI Sex",
    # "ppmi_age": "PPMI Age",
    "aabc_sex": "HCP-A Sex",
    "aabc_age": "HCP-A Age",
    "hcpya_rest1lr_gender": "HCP-YA Sex",
    # "hcpya_rest1lr_age": "HCP-YA Age",
    # "hcpya_rest1lr_neofacn": "HCP-YA NEO-N",
    "hcpya_task21": "HCP-YA Task21",
    "nsd_cococlip": "NSD COCO24",
}

In [8]:
dataset_table = []
for key, label in tqdm(DATASET_NAMES.items()):
    name, target = label.split()
    dataset = create_dataset(key, "schaefer400")
    info = get_dataset_info(dataset)

    record = {"name": name, "target": target, **info}
    dataset_table.append(record)

dataset_table = pd.DataFrame.from_records(dataset_table)
print(dataset_table.to_markdown(index=False))

100%|██████████| 13/13 [01:38<00:00,  7.61s/it]

| name    | target   |   subjects |   samples |   sample length | sample tr   | splits   |   num classes | majority ratio   | labels                                                                                                                                                                                                              |
|:--------|:---------|-----------:|----------:|----------------:|:------------|:---------|--------------:|:-----------------|:--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| ABIDE   | Dx       |        826 |       826 |             150 | 2.0s        | 70:15:15 |             2 | 55%              | Autism: 371 Control: 455                                                                                                                                                                          

In [9]:
dataset_table_fmt = dataset_table.iloc[[0, 2, 4, 6, 9, 10, 11, 12], :9]
dataset_table_fmt.columns = [
    "Dataset",
    "Target",
    "Subjects",
    "Samples",
    "Sample length",
    "TR",
    "Splits (train/val/test)",
    "\#Classes",
    "Majority class",
]
print(dataset_table_fmt.to_latex(index=False))

\begin{tabular}{llrrlllrl}
\toprule
Dataset & Target & Subjects & Samples & Sample length & TR & Splits (train/val/test) & \#Classes & Majority class \\
\midrule
ABIDE & Dx & 826 & 826 & 150 & 2.0s & 70:15:15 & 2 & 55% \\
ADHD200 & Dx & 430 & 430 & 150 & 2.0s & 70:15:15 & 2 & 57% \\
ADNI & Dx & 410 & 410 & 100 & 3.0s & 80:10:10 & 2 & 77% \\
PPMI & Dx & 662 & 662 & 120 & 2.5s & 70:15:15 & 2 & 62% \\
HCP-A & Age & 560 & 560 & 500 & 0.7s & 81:9:9 & 4 & 27% \\
HCP-YA & Sex & 598 & 598 & 500 & 0.7s & 68:15:17 & 2 & 50% \\
HCP-YA & Task21 & 614 & 28071 & 16 & 1.0s & 68:14:18 & 21 & 17% \\
NSD & COCO24 & 8 & 43347 & 16 & 1.0s & 75:12:12 & 24 & 7% \\
\bottomrule
\end{tabular}



In [4]:
for key in DATASET_NAMES:
    dataset = create_dataset(key, "schaefer400")

    split_subs = {}
    for split, ds in dataset.items():
        subs = np.asarray(ds.dataset["sub"])[ds.indices]
        subs, counts = np.unique(subs, return_counts=True)
        if counts.max() > 1:
            print(key, split, counts.max().item())
        split_subs[split] = subs

    splits = list(split_subs)
    for ii, split_a in enumerate(splits):
        for split_b in splits[ii + 1 :]:
            n_overlap = len(np.intersect1d(split_subs[split_a], split_subs[split_b]))
            if n_overlap > 0:
                print(key, split_a, split_b, n_overlap)

hcpya_task21 train 49
hcpya_task21 validation 49
hcpya_task21 test 49
nsd_cococlip train 6236
nsd_cococlip validation 5418
nsd_cococlip test 5390
nsd_cococlip testid 885
nsd_cococlip train testid 6
